# Environment

In [ ]:
# FLAML import
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import os
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, f1_score, fbeta_score, 
                             precision_score, recall_score, confusion_matrix, 
                             classification_report, average_precision_score)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt

import data_preprocessing.kbase as kb 

# Data Loading

In [ ]:
# Loading exported table
df_triaje_stroke = pq.read_table(os.path.join(kb.source_tables_path, "df_modelling/df_triaje_cleaned.parquet")).to_pandas()
df_triaje_stroke.shape

In [ ]:
df_triaje_stroke.info()

In [ ]:
df_triaje_stroke.head(5)

# Data and column filtering

In [ ]:
# Filter specific columns for modeling
selected_cols = [col for col in kb.modelling_cols if col in df_triaje_stroke.columns]
df_triaje_stroke = df_triaje_stroke[selected_cols]
nrow_triaje = len(df_triaje_stroke)
print("Total rows before cohort filtering:", nrow_triaje)

# Filter por stroke code (54)
df_triaje_stroke = df_triaje_stroke[df_triaje_stroke["demand_type_1"] == 54].copy()
df_triaje_stroke = df_triaje_stroke.drop(columns = ["demand_type_1"])
nrow_triaje_stroke = len(df_triaje_stroke)
print("Rows after stroke filtering:", len(df_triaje_stroke))

# Filter for adult patients
df_triaje_stroke = df_triaje_stroke[df_triaje_stroke["age"] >= 18].copy()
nrow_triaje_stroke_adults = len(df_triaje_stroke)
print("Total rows after filtering adult patients", nrow_triaje_stroke_adults)


In [ ]:
df_triaje_stroke.info()

In [ ]:
df_triaje_stroke["p1_real"].value_counts(dropna=False)

# User configuration

In [ ]:
# USER CONFIGURATION (EDIT THIS SECTION)
# Imports & Configuration
# CSV_FILE = "dataset.csv"       # Filename of the dataset
TARGET_COLUMN = "p1_real"       # Name of the column to predict
TIME_BUDGET = 600              # Training time in seconds (e.g., 600 = 10 min)
TEST_SIZE = 0.2                # 20% for testing
SEED = 42
#---

# Data split

In [ ]:
# 1. LOAD DATA & SPLIT

try:
    # Validate target column exists
    if TARGET_COLUMN not in df_triaje_stroke.columns:
        raise ValueError(f"Column '{TARGET_COLUMN}' not found in dataset.")

    # Separate Features (X) and Target (y)
    X = df_triaje_stroke.drop(columns=[TARGET_COLUMN])
    y = df_triaje_stroke[TARGET_COLUMN]

    # Split into Train and Test sets
    # stratify=y ensures both sets have the same proportion of positive cases
    print("--- Splitting Data ---")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples:  {len(X_test)}")

except Exception as e:
    print(f"Error loading data: {e}")

In [ ]:
# 2. TRAINING FLAML
# Configures the AutoML engine and starts training.
print(f"\n--- Starting AutoML (Budget: {TIME_BUDGET}s) ---")

# Calculate sample weights:
# This gives more importance to the minority class (events) during training.
# Critical for medical datasets where events are rare.
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

# Initialize AutoML
automl = AutoML()
automl_settings = {
    "time_budget": TIME_BUDGET, # Time limit in seconds
    "metric": "auto",           # 'auto' lets FLAML choose the best metric (usually ROC_AUC or LogLoss)
    "task": "classification",   # We are predicting categories
    "eval_method": "cv",        # Use Cross-Validation
    "n_splits": 5,              # 5-Fold CV
    "seed": SEED,
    "verbose": 1,               # Keep console output clean
    "log_training_metric": True,
}

# Train the model
automl.fit(X_train=X_train, y_train=y_train, sample_weight=sample_weight, **automl_settings)

print("\n--- Training Completed ---")
print(f"Best Estimator Found: {automl.best_estimator}")
print(f"Best CV Score: {1 - automl.best_loss:.4f}")

In [ ]:
# 3. PREDICTIONS & THRESHOLD OPTIMIZATION
# Instead of using the default 0.5 cutoff, we find a threshold that prioritizes finding positive cases.
print("\n--- Optimizing Decision Threshold (F2-Score) ---")

# Get raw probabilities (0.0 to 1.0) for Train and Test
y_train_prob = automl.predict_proba(X_train)[:, 1]
y_test_prob = automl.predict_proba(X_test)[:, 1]

# We search for the best threshold using F2-Score.
# F2-Score weights Recall higher than Precision. 
# This is better for detecting rare events (we prefer false alarms over missing a case).
thresholds = np.arange(0.05, 0.95, 0.01)
f2_scores = [fbeta_score(y_train, (y_train_prob >= t).astype(int), beta=2) for t in thresholds]

# Select the threshold that gave the max F2 score on the training set
best_thr = thresholds[np.argmax(f2_scores)]

print(f"Optimal Threshold found: {best_thr:.3f}")

In [ ]:
# 4. TEST SET RESULTS
# Applies the optimized threshold to the Test set and reports performance.
print("\n--- Test Set Results ---")

# Convert probabilities to binary predictions using our custom threshold
y_test_pred = (y_test_prob >= best_thr).astype(int)

# Calculate global metrics
roc_auc = roc_auc_score(y_test, y_test_prob)
ap = average_precision_score(y_test, y_test_prob) # Area Under Precision-Recall Curve
f1 = f1_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)       # Sensitivity (How many events did we catch?)
precision = precision_score(y_test, y_test_pred) # (How many predictions were correct?)

# Confusion Matrix components
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()

print(f"ROC-AUC:       {roc_auc:.4f}")
print(f"PR-AUC:        {ap:.4f}")
print(f"F1-Score:      {f1:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall (Sens): {recall:.4f}")
print("-" * 30)
print(f"Confusion Matrix:\nTN: {tn} (True Neg) | FP: {fp} (False Alarm)\nFN: {fn} (Missed)   | TP: {tp} (Hit)")
print("-" * 30)
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

In [ ]:
# 5. FEATURE IMPORTANCE
# Visualizes which variables influenced the model the most.
print("\n--- Feature Importance (Permutation) ---")

try:
    # We use Permutation Importance:
    # It shuffles each column and checks how much the model performance drops.
    # If performance drops a lot, the feature is important.
    perm_imp = permutation_importance(
        automl, X_test, y_test, 
        scoring='average_precision', # Optimized for imbalanced classes
        n_repeats=5, 
        random_state=42, 
        n_jobs=-1
    )
    
    # Organize results into a DataFrame
    imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': perm_imp.importances_mean})
    imp_df = imp_df.sort_values(by='Importance', ascending=False).head(15)
    
    print("Top Features:\n", imp_df.to_string(index=False))
    
    # Plotting
    plt.figure(figsize=(10, 6))
    plt.barh(imp_df['Feature'][::-1], imp_df['Importance'][::-1], color='#333333')
    plt.title('Top 15 Features (Permutation Importance)')
    plt.xlabel('Importance (Impact on Model Performance)')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"Feature importance skipped: {e}")